### Projection

In [1]:
import os
import sys
import glob
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

import jax
import jax.numpy as jnp

from tqdm import tqdm

# ============================================================
# PATHS
# ============================================================
main_dir = "/data/keeling/a/tahsina2/Alam_et_al_2026" #Update
LOCA2_DIR = "/data/keeling/a/tahsina2/a/LOCA2" #Update
OUT_DIR = "/data/keeling/a/tahsina2/a/260425_Projection" #Update
PROJ_ROOT = "/data/keeling/a/tahsina2/a/260425_Projection" #Update
DATA_DIR = f"{main_dir"}/data"
loca_dir = Path("/data/keeling/a/tahsina2/a/LOCA2")

MISSING_CSV = f"{DATA_DIR}/missing_raw_projection_tasks.csv"
MISSING_TXT = f"{DATA_DIR}/missing_raw_projection_task_ids.txt"
EXPECTED_CSV = f"{DATA_DIR}/expected_raw_projection_tasks.csv"

METRIC = "rmse"

PROJ_BASE_DIR = os.path.join(PROJ_ROOT, METRIC)

HIST_NC = f"{DATA_DIR}/Historical_Inputs_biascorrected.nc"


SRC_DIR  = f"{main_dir}/src"
sys.path.append(SRC_DIR)
# import your model functions
from water_balance_jax import (
    wbm_jax,
    construct_Kpet_gen,
    construct_Kpet_crop,
)

### Data Validation Script

In [ ]:
# ============================================================
# SETTINGS
# ============================================================

CALIB_DIRS = {
    "kgeprime":    f"{DATA_DIR}/Calibration/AllStations/calibration_outputs_kgeprime_gridmet",
    "rmse":        f"{DATA_DIR}/Calibration/AllStations/calibration_outputs_rmse_gridmet",
    "outer50rmse": f"{DATA_DIR}/Calibration/AllStations/calibration_outputs_outer50rmse_gridmet",
    "kge":         f"{DATA_DIR}/Calibration/AllStations/calibration_outputs_kge_gridmet",
}

METRIC_ORDER = ["kgeprime", "rmse", "outer50rmse", "kge"]
N_SCENARIOS = 170

KEEP_RAW = {
    "smap_raw",
    "vic_raw",
    "noah_raw",
    "mosaic_raw",
}

# ============================================================
# HELPERS
# ============================================================

def get_loca2_models():
    zarrs = sorted(glob.glob(os.path.join(LOCA2_DIR, "*.zarr")))

    if len(zarrs) == 0:
        raise FileNotFoundError(f"No .zarr files found in {LOCA2_DIR}")

    return zarrs


def parse_calibration_filename(calib_file):
    """
    Expected format:
    scenario_0040_BRW_NOAH_raw_20160101.nc

    Returns:
    scenario_id, station, scenario_label
    """

    fname = os.path.basename(calib_file)

    pattern = (
        r"^scenario_(\d{4})_"
        r"([A-Za-z0-9]+)_"
        r"([A-Za-z0-9]+)_"
        r"(raw|bc)_"
        r"\d{8}\.nc$"
    )

    m = re.match(pattern, fname)

    if m is None:
        raise ValueError(f"Could not parse calibration filename: {fname}")

    scenario_id = int(m.group(1))
    station = m.group(2)
    source = m.group(3).lower()
    raw_or_bc = m.group(4).lower()

    scenario_label = f"{source}_{raw_or_bc}"

    return scenario_id, station, scenario_label


def get_calibration_lookup(metric_name):
    """
    Build lookup:
    scenario_id -> metadata parsed from filename
    """

    files = sorted(
        glob.glob(
            os.path.join(
                CALIB_DIRS[metric_name],
                "scenario_*.nc"
            )
        )
    )

    lookup = {}

    for f in files:
        try:
            scenario_id, station, scenario_label = parse_calibration_filename(f)
        except Exception:
            continue

        if scenario_label not in KEEP_RAW:
            continue

        if scenario_id in lookup:
            raise RuntimeError(
                f"Duplicate scenario_id={scenario_id} for metric={metric_name}: "
                f"{lookup[scenario_id]['calibration_file']} and {f}"
            )

        lookup[scenario_id] = {
            "scenario_id": scenario_id,
            "station": station,
            "scenario_label": scenario_label,
            "calibration_file": f,
        }

    return lookup


def make_task_id(model_idx, metric_idx, scenario_id):
    return (
        model_idx * (N_SCENARIOS * len(METRIC_ORDER))
        + metric_idx * N_SCENARIOS
        + scenario_id
    )


# ============================================================
# LOAD MODELS AND CALIBRATION LOOKUPS
# ============================================================

loca2_models = get_loca2_models()

print(f"LOCA2 models found: {len(loca2_models)}")

calib_lookup_by_metric = {}

for metric_name in METRIC_ORDER:
    calib_lookup_by_metric[metric_name] = get_calibration_lookup(metric_name)

    print(
        f"{metric_name:12s}: "
        f"{len(calib_lookup_by_metric[metric_name])} raw calibration scenarios"
    )

# ============================================================
# BUILD EXPECTED RAW TASK TABLE
# ============================================================

expected_rows = []

total_iterations = sum(
    len(loca2_models) * len(calib_lookup_by_metric[m])
    for m in METRIC_ORDER
)

pbar = tqdm(
    total=total_iterations,
    desc="Checking expected raw projection files",
)

for model_idx, zarr_path in enumerate(loca2_models):

    model_name = os.path.basename(zarr_path).replace(".zarr", "")

    for metric_idx, metric_name in enumerate(METRIC_ORDER):

        lookup = calib_lookup_by_metric[metric_name]

        for scenario_id, meta in lookup.items():

            pbar.update(1)

            scenario_label = meta["scenario_label"]
            station = meta["station"]
            calib_file = meta["calibration_file"]

            task_id = make_task_id(
                model_idx=model_idx,
                metric_idx=metric_idx,
                scenario_id=scenario_id,
            )

            expected_file = os.path.join(
                OUT_DIR,
                metric_name,
                model_name,
                f"scenario_{scenario_id:04d}_{scenario_label}",
                f"{station}_projection_2023_2100.nc",
            )

            expected_rows.append({
                "task_id": task_id,
                "model_idx": model_idx,
                "model_name": model_name,
                "metric_idx": metric_idx,
                "metric": metric_name,
                "scenario_id": scenario_id,
                "scenario_label": scenario_label,
                "station": station,
                "expected_file": expected_file,
                "calibration_file": calib_file,
                "exists": os.path.exists(expected_file),
            })

pbar.close()

# ============================================================
# DATAFRAME
# ============================================================

expected_df = pd.DataFrame(expected_rows)
expected_df.to_csv(EXPECTED_CSV, index=False)

print(f"\nExpected raw projection rows: {len(expected_df):,}")
print(f"Saved expected table: {EXPECTED_CSV}")

# ============================================================
# SUMMARIES
# ============================================================

print("\n====================================================")
print("EXPECTED RAW TASKS BY SOURCE + METRIC")
print("====================================================")
display(expected_df.groupby(["scenario_label", "metric"]).size())

print("\n====================================================")
print("EXPECTED RAW TASKS BY SOURCE")
print("====================================================")
display(expected_df.groupby("scenario_label").size())

print("\n====================================================")
print("EXPECTED UNIQUE SCENARIO IDS BY SOURCE + METRIC")
print("====================================================")
display(
    expected_df.groupby(
        ["scenario_label", "metric"]
    )["scenario_id"].nunique()
)

# ============================================================
# FIND MISSING
# ============================================================

missing_df = expected_df[~expected_df["exists"]].copy()

missing_df.to_csv(MISSING_CSV, index=False)

missing_df["task_id"].to_csv(
    MISSING_TXT,
    index=False,
    header=False
)

# ============================================================
# COMPLETION SUMMARY
# ============================================================

print("\n====================================================")
print("COMPLETION SUMMARY")
print("====================================================")

n_expected = len(expected_df)
n_missing = len(missing_df)
n_done = n_expected - n_missing

print(f"Expected raw projection files: {n_expected:,}")
print(f"Completed raw projection files: {n_done:,}")
print(f"Missing raw projection files:   {n_missing:,}")

if n_expected > 0:
    print(f"Completion rate: {100 * n_done / n_expected:.2f}%")

print(f"\nSaved missing CSV: {MISSING_CSV}")
print(f"Saved missing task IDs: {MISSING_TXT}")

# ============================================================
# MISSING SUMMARIES
# ============================================================

print("\n====================================================")
print("MISSING BY SOURCE + METRIC")
print("====================================================")
if len(missing_df) > 0:
    display(missing_df.groupby(["scenario_label", "metric"]).size())
else:
    print("No missing raw projections.")

print("\n====================================================")
print("MISSING BY SOURCE")
print("====================================================")
if len(missing_df) > 0:
    display(missing_df.groupby("scenario_label").size())
else:
    print("No missing raw projections.")

print("\n====================================================")
print("MISSING BY METRIC")
print("====================================================")
if len(missing_df) > 0:
    display(missing_df.groupby("metric").size())
else:
    print("No missing raw projections.")

print("\n====================================================")
print("MISSING UNIQUE SCENARIO IDS BY SOURCE + METRIC")
print("====================================================")
if len(missing_df) > 0:
    display(
        missing_df.groupby(
            ["scenario_label", "metric"]
        )["scenario_id"].nunique()
    )
else:
    print("No missing raw projections.")

# ============================================================
# SANITY CHECK
# ============================================================

bad_labels = sorted(set(expected_df["scenario_label"]) - KEEP_RAW)

print("\n====================================================")
print("SANITY CHECK")
print("====================================================")

if bad_labels:
    print("Unexpected labels found:")
    print(bad_labels)
else:
    print("Only raw SMAP/VIC/NOAH/MOSAIC labels included.")

# ============================================================
# SHOW FIRST FEW MISSING TASKS
# ============================================================

if len(missing_df) > 0:
    print("\nFirst 20 missing tasks:")
    display(missing_df.head(20))

In [ ]:

# -------------------------
# Find all .zarr projection folders
# Example:
# ACCESS-CM2_r1i1p1f1_ssp245.zarr
# -------------------------
zarr_paths = sorted(loca_dir.glob("*.zarr"))

records = []

pattern = re.compile(
    r"^(?P<gcm>.+?)_(?P<realization>r\d+i\d+p\d+f\d+)_(?P<ssp>ssp\d+)\.zarr$"
)

for path in zarr_paths:
    match = pattern.match(path.name)

    if match is None:
        print(f"Skipping unmatched folder name: {path.name}")
        continue

    records.append({
        "path": str(path),
        "filename": path.name,
        "gcm": match.group("gcm"),
        "realization": match.group("realization"),
        "ssp": match.group("ssp")
    })

df = pd.DataFrame(records)

# -------------------------
# Basic summary
# -------------------------
print("Total compatible projections:", len(df))
print("Number of unique GCMs:", df["gcm"].nunique())
print("SSPs used:", sorted(df["ssp"].unique()))
print("Number of unique realizations:", df["realization"].nunique())

print("\nNumber of projections per SSP:")
print(df["ssp"].value_counts().sort_index())

print("\nNumber of GCMs per SSP:")
print(df.groupby("ssp")["gcm"].nunique())

print("\nNumber of projections per GCM:")
print(df["gcm"].value_counts().sort_index())

# -------------------------
# SSP x GCM table
# -------------------------
ssp_gcm_table = pd.crosstab(df["gcm"], df["ssp"])

print("\nGCM by SSP projection count:")
print(ssp_gcm_table)

# -------------------------
# Realizations per GCM and SSP
# -------------------------
realization_table = (
    df.groupby(["gcm", "ssp"])["realization"]
    .nunique()
    .reset_index()
    .rename(columns={"realization": "n_realizations"})
)

print("\nRealizations per GCM and SSP:")
print(realization_table)

# -------------------------
# Check why total equals 201
# -------------------------
total_from_table = ssp_gcm_table.to_numpy().sum()

print("\nCheck total from GCM x SSP table:", total_from_table)

# -------------------------
# Save summaries
# -------------------------
out_dir = DATA_DIR
df.to_csv(out_dir / "loca2_projection_inventory.csv", index=False)
ssp_gcm_table.to_csv(out_dir / "loca2_gcm_by_ssp_table.csv")
realization_table.to_csv(out_dir / "loca2_realizations_per_gcm_ssp.csv", index=False)

print("\nSaved:")
print(out_dir / "loca2_projection_inventory.csv")
print(out_dir / "loca2_gcm_by_ssp_table.csv")
print(out_dir / "loca2_realizations_per_gcm_ssp.csv")

### Data Preparation for Seasonal Cycle Plots

In [1]:
#!/usr/bin/env python3


# ============================================================
# SETTINGS
# ============================================================
METRICS = ["kgeprime", "rmse", "outer50rmse", "kge"]

GROUPS = [
    #"smap_raw",
    #"insitu7",
    "noah_raw",
    "vic_raw",
    "mosaic_raw",
]

GROUP_LABELS = {
    "smap_raw": "SMAP raw",
    "insitu7": "In-situ 2016-2022",
    "noah_raw": "Noah raw",
    "vic_raw": "VIC raw",
    "mosaic_raw": "Mosaic raw",
}

HIST_SOURCE_MAP = {
    "smap_raw": ("SMAP_sm", 2016, 2022),
    "insitu7": ("Insitu_sm", 2016, 2022),
    "noah_raw": ("NOAH_sm", 2016, 2022),
    "vic_raw": ("VIC_sm", 2016, 2022),
    "mosaic_raw": ("MOSAIC_sm", 2016, 2022),
}

PERIODS = {
    "2030s": (2030, 2039),
    "2060s": (2060, 2069),
    "2090s": (2090, 2099),
}

MONTHS = np.arange(1, 13)

EXCLUDE_STATIONS = set()

# ============================================================
# HELPERS
# ============================================================
def open_nc_safe(path):
    last_err = None

    for eng in ["netcdf4", "h5netcdf", "scipy"]:
        try:
            ds = xr.open_dataset(path, engine=eng)
            _ = ds.dims
            return ds
        except Exception as e:
            last_err = e

    raise RuntimeError(f"Could not open {path}. Last error: {last_err}")


def get_station_list(hist_nc):
    ds = open_nc_safe(hist_nc)

    try:
        stations = [
            str(s)
            for s in ds["station"].values
            if str(s) not in EXCLUDE_STATIONS
        ]
    finally:
        ds.close()

    return sorted(stations)


def get_period_mask(time_coord, start_year, end_year):
    return (time_coord.dt.year >= start_year) & (time_coord.dt.year <= end_year)


def monthly_stats_from_values(values):
    values = np.asarray(values, dtype=float).ravel()
    values = values[np.isfinite(values)]

    if values.size == 0:
        return {
            "mean": np.nan,
            "median": np.nan,
            "q25": np.nan,
            "q75": np.nan,
            "min": np.nan,
            "max": np.nan,
            "n": 0,
        }

    return {
        "mean": float(np.nanmean(values)),
        "median": float(np.nanmedian(values)),
        "q25": float(np.nanpercentile(values, 25)),
        "q75": float(np.nanpercentile(values, 75)),
        "min": float(np.nanmin(values)),
        "max": float(np.nanmax(values)),
        "n": int(values.size),
    }


def monthly_pooled_stats(da, start_year, end_year):
    mask = get_period_mask(da["time"], start_year, end_year)
    da = da.where(mask, drop=True)

    rows = []

    for month in MONTHS:
        vals = da.where(da["time.month"] == month, drop=True).values
        stats = monthly_stats_from_values(vals)

        row = {"month": int(month)}
        row.update(stats)
        rows.append(row)

    return pd.DataFrame(rows)


def list_projection_files(metric, group, stations):
    base_dir = os.path.join(PROJ_ROOT, metric)

    pattern = os.path.join(
        base_dir,
        "*",
        f"scenario_*_{group}",
        "*_projection_2023_2100.nc"
    )

    files = sorted(glob.glob(pattern))

    rows = []

    for f in files:
        station = os.path.basename(f).replace("_projection_2023_2100.nc", "")

        if station not in stations:
            continue

        parts = f.split(os.sep)
        model = parts[-3]
        scenario_dir = parts[-2]

        rows.append({
            "metric": metric,
            "model": model,
            "group": group,
            "group_label": GROUP_LABELS[group],
            "scenario_dir": scenario_dir,
            "station": station,
            "file": f,
        })

    return pd.DataFrame(rows)


# ============================================================
# HISTORICAL MONTHLY STATS
# ============================================================
def build_historical_monthly_stats(hist_nc, stations):
    ds = open_nc_safe(hist_nc)
    records = []

    try:
        print("\nHistorical variables available:")
        print(list(ds.data_vars))

        for group in GROUPS:
            varname, start_year, end_year = HIST_SOURCE_MAP[group]

            if varname not in ds.data_vars:
                print(f"WARNING: {varname} not found. Skipping historical {group}.")
                continue

            da = ds[varname].sel(station=stations)
            monthly_df = monthly_pooled_stats(da, start_year, end_year)

            for row in monthly_df.itertuples(index=False):
                records.append({
                    "source": "historical",
                    "metric": "historical",
                    "group": group,
                    "group_label": GROUP_LABELS[group],
                    "period": "historical",
                    "month": int(row.month),
                    "mean": float(row.mean),
                    "median": float(row.median),
                    "q25": float(row.q25),
                    "q75": float(row.q75),
                    "min": float(row.min),
                    "max": float(row.max),
                    "n": int(row.n),
                    "n_models": 0,
                    "n_stations": int(len(stations)),
                })

    finally:
        ds.close()

    return pd.DataFrame(records)


# ============================================================
# PROJECTION MONTHLY STATS
# ============================================================
def build_projection_monthly_stats_for_metric(metric, stations):
    records = []

    for group in GROUPS:
        file_table = list_projection_files(metric, group, stations)

        print(f"{metric} | {group}: {len(file_table)} projection files")

        if file_table.empty:
            continue

        for period_name, (start_year, end_year) in PERIODS.items():
            month_values = {m: [] for m in MONTHS}
            models_used = set()
            stations_used = set()

            for row in tqdm(
                file_table.itertuples(index=False),
                total=len(file_table),
                desc=f"{metric} {group} {period_name}"
            ):
                ds = None

                try:
                    ds = open_nc_safe(row.file)

                    if "soil_moisture" not in ds.data_vars:
                        print(f"Skipping {row.file}: soil_moisture not found")
                        continue

                    sm = ds["soil_moisture"]
                    mask = get_period_mask(sm["time"], start_year, end_year)
                    sm_period = sm.where(mask, drop=True)

                    if sm_period.sizes.get("time", 0) == 0:
                        continue

                    models_used.add(row.model)
                    stations_used.add(row.station)

                    for month in MONTHS:
                        vals = sm_period.where(
                            sm_period["time.month"] == month,
                            drop=True
                        ).values

                        vals = np.asarray(vals, dtype=float).ravel()
                        vals = vals[np.isfinite(vals)]

                        if vals.size > 0:
                            month_values[month].append(vals)

                except Exception as e:
                    print(f"Skipping {row.file}: {repr(e)}")

                finally:
                    if ds is not None:
                        ds.close()

            for month in MONTHS:
                if len(month_values[month]) == 0:
                    vals = np.array([], dtype=float)
                else:
                    vals = np.concatenate(month_values[month])

                stats = monthly_stats_from_values(vals)

                records.append({
                    "source": "projection",
                    "metric": metric,
                    "group": group,
                    "group_label": GROUP_LABELS[group],
                    "period": period_name,
                    "month": int(month),
                    "mean": float(stats["mean"]),
                    "median": float(stats["median"]),
                    "q25": float(stats["q25"]),
                    "q75": float(stats["q75"]),
                    "min": float(stats["min"]),
                    "max": float(stats["max"]),
                    "n": int(stats["n"]),
                    "n_models": int(len(models_used)),
                    "n_stations": int(len(stations_used)),
                })

    return pd.DataFrame(records)


# ============================================================
# SAVE TO NETCDF
# ============================================================
def save_summary_to_netcdf(df, out_nc):
    ds = (
        df.set_index(["source", "metric", "group", "period", "month"])
        .to_xarray()
        .sortby("month")
    )

    ds.attrs["description"] = (
        "Monthly soil moisture cycle statistics pooled across all 17 stations. "
        "Includes historical and projected periods for SMAP raw, in-situ, Noah raw, VIC raw, and Mosaic raw."
    )
    ds.attrs["projection_root"] = PROJ_ROOT
    ds.attrs["historical_file"] = HIST_NC
    ds.attrs["groups"] = ", ".join(GROUPS)
    ds.attrs["metrics"] = ", ".join(METRICS)
    ds.attrs["periods"] = ", ".join(
        [f"{k}:{v[0]}-{v[1]}" for k, v in PERIODS.items()]
    )
    ds.attrs["historical_periods"] = (
        "smap_raw=2016-2022 using SMAP_sm; "
        "insitu7=2016-2022 using Insitu_sm; "
        "noah_raw=2016-2022 using NOAH_sm; "
        "vic_raw=2016-2022 using VIC_sm; "
        "mosaic_raw=2016-2022 using MOSAIC_sm"
    )

    ds.to_netcdf(out_nc, engine="netcdf4")


# ============================================================
# MAIN
# ============================================================
def main():
    stations = get_station_list(HIST_NC)

    print(f"Stations used ({len(stations)}):")
    print(stations)

    hist_df = build_historical_monthly_stats(HIST_NC, stations)

    all_records = []

    for metric in METRICS:
        print("\n" + "=" * 80)
        print(f"Processing metric: {metric}")
        print("=" * 80)

        proj_df = build_projection_monthly_stats_for_metric(metric, stations)

        metric_df = pd.concat(
            [
                hist_df.assign(metric=metric),
                proj_df,
            ],
            ignore_index=True
        )

        out_csv = os.path.join(
            OUT_DIR,
            f"soil_moisture_monthly_cycle_all_raw_all17_{metric}.csv"
        )

        out_nc = os.path.join(
            OUT_DIR,
            f"soil_moisture_monthly_cycle_all_raw_all17_{metric}.nc"
        )

        metric_df.to_csv(out_csv, index=False)
        save_summary_to_netcdf(metric_df, out_nc)

        print(f"Saved CSV: {out_csv}")
        print(f"Saved NC:  {out_nc}")

        all_records.append(metric_df)

    combined_df = pd.concat(all_records, ignore_index=True)

    combined_csv = os.path.join(
        OUT_DIR,
        "soil_moisture_monthly_cycle_all_raw_all17_all_metrics.csv"
    )

    combined_nc = os.path.join(
        OUT_DIR,
        "soil_moisture_monthly_cycle_all_raw_all17_all_metrics.nc"
    )

    combined_df.to_csv(combined_csv, index=False)
    save_summary_to_netcdf(combined_df, combined_nc)

    print("\nDone.")
    print(f"Combined CSV: {combined_csv}")
    print(f"Combined NC:  {combined_nc}")

    print("\nQuick count check:")
    print(
        combined_df.groupby(["metric", "source", "group", "period"])["n_models"]
        .max()
        .reset_index()
        .to_string(index=False)
    )


if __name__ == "__main__":
    main()

Stations used (17):
['BRW', 'BVL', 'CMI', 'DEK', 'DXS', 'FAI', 'FRE', 'FRM', 'ICC', 'LLC', 'MON', 'OLN', 'ORR', 'RND', 'SIU', 'STC', 'STE']

Historical variables available:
['Insitu_sm', 'Gridmet_tas', 'Gridmet_pr', 'Nldas_tas', 'Nldas_pr', 'Smap_tas', 'Smap_pr', 'VIC_sm', 'MOSAIC_sm', 'NOAH_sm', 'SMAP_sm', 'VIC_sm_bc', 'MOSAIC_sm_bc', 'NOAH_sm_bc', 'SMAP_sm_bc']

Processing metric: kgeprime
kgeprime | noah_raw: 3417 projection files


kgeprime noah_raw 2090s: 100%|██████████████| 3417/3417 [02:54<00:00, 19.54it/s]


kgeprime | vic_raw: 3417 projection files


kgeprime vic_raw 2090s: 100%|███████████████| 3417/3417 [02:45<00:00, 20.66it/s]


kgeprime | mosaic_raw: 3417 projection files


...

kge | vic_raw: 3417 projection files


kge vic_raw 2090s: 100%|████████████████████| 3417/3417 [02:44<00:00, 20.73it/s]


kge | mosaic_raw: 3417 projection files


kge mosaic_raw 2090s: 100%|█████████████████| 3417/3417 [02:53<00:00, 19.73it/s]


Saved CSV: /data/keeling/a/tahsina2/Alam_et_al_2026/data/processed_projection_stats_monthly_cycle_all17_all_raw/soil_moisture_monthly_cycle_all_raw_all17_kge.csv
Saved NC:  /data/keeling/a/tahsina2/Alam_et_al_2026/data/processed_projection_stats_monthly_cycle_all17_all_raw/soil_moisture_monthly_cycle_all_raw_all17_kge.nc

Done.
Combined CSV: /data/keeling/a/tahsina2/Alam_et_al_2026/data/processed_projection_stats_monthly_cycle_all17_all_raw/soil_moisture_monthly_cycle_all_raw_all17_all_metrics.csv
Combined NC:  /data/keeling/a/tahsina2/Alam_et_al_2026/data/processed_projection_stats_monthly_cycle_all17_all_raw/soil_moisture_monthly_cycle_all_raw_all17_all_metrics.nc

Quick count check:
     metric     source      group     period  n_models
        kge historical mosaic_raw historical         0
        kge historical   noah_raw historical         0
        kge historical    vic_raw historical         0
        kge projection mosaic_raw      2030s       201
        kge projection mosaic_

In [4]:
#!/usr/bin/env python3

# Keep same OUT_DIR as before
OUT_DIR = f"{DATA_DIR"}/processed_projection_stats_precip_monthly_totals"
os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================
# SETTINGS
# ============================================================
METRIC = "raw"

FILELEVEL_CSV = os.path.join(
    OUT_DIR,
    f"precip_monthly_totals_filelevel_with_historical_{METRIC}_gridmet_loca2.csv"
)

LOG_JSON = os.path.join(
    OUT_DIR,
    f"completed_precip_monthly_totals_{METRIC}_gridmet_loca2.json"
)

SUMMARY_NC = os.path.join(
    OUT_DIR,
    f"precip_monthly_totals_with_historical_{METRIC}_gridmet_loca2.nc"
)

GROUPS = ["loca2"]

GROUP_LABELS = {
    "loca2": "LOCA2 projection",
    "gridmet": "GRIDMET historical",
}

GRIDMET_GROUP = "gridmet"
GRIDMET_PERIOD = (2016, 2022)

PERIODS = {
    "2030s": (2030, 2039),
    "2060s": (2060, 2069),
    "2090s": (2090, 2099),
}

MONTHS = np.arange(1, 13)

EXCLUDE_STATIONS = set()

GRIDMET_PR_CANDIDATES = [
    "Gridmet_pr",
    "gridmet_pr",
    "Gridmet_prcp",
    "gridmet_prcp",
    "prcp",
    "pr",
]

LOCA2_PR_CANDIDATES = [
    "pr",
    "precip",
    "precipitation",
    "prcp",
]


# ============================================================
# HELPERS
# ============================================================
def open_nc_safe(path):
    last_err = None

    for eng in ["netcdf4", "h5netcdf", "scipy"]:
        try:
            ds = xr.open_dataset(path, engine=eng)
            _ = ds.dims
            return ds
        except Exception as e:
            last_err = e

    raise RuntimeError(f"Could not open {path}. Last error: {last_err}")


def get_period_mask(time_coord, start_year, end_year):
    return (time_coord.dt.year >= start_year) & (time_coord.dt.year <= end_year)


def get_station_list(hist_nc):
    ds = open_nc_safe(hist_nc)

    try:
        stations = [
            str(s)
            for s in ds["station"].values
            if str(s) not in EXCLUDE_STATIONS
        ]
    finally:
        ds.close()

    return sorted(stations)


def find_var(ds, candidates, label):
    for v in candidates:
        if v in ds.data_vars:
            return v

    raise KeyError(
        f"Could not find {label} variable. "
        f"Tried {candidates}. "
        f"Available data_vars: {list(ds.data_vars)}"
    )


def monthly_total_climatology_values(da, mask):
    da_period = da.where(mask, drop=True)

    if da_period.sizes.get("time", 0) == 0:
        return None

    vals = []

    for month in MONTHS:
        da_month = da_period.where(da_period["time.month"] == month, drop=True)

        if da_month.sizes.get("time", 0) == 0:
            vals.append(np.nan)
        else:
            monthly_totals = da_month.groupby("time.year").sum("time", skipna=True)
            vals.append(float(monthly_totals.mean("year", skipna=True).values))

    return vals


def list_loca2_projection_files(stations):
    pattern = os.path.join(
        PROJ_ROOT,
        "*",
        "*",
        "scenario_*",
        "*_projection_2023_2100.nc"
    )

    files = sorted(glob.glob(pattern))
    rows = []

    for f in files:
        station = os.path.basename(f).replace("_projection_2023_2100.nc", "")

        if station not in stations:
            continue

        parts = f.split(os.sep)

        scenario_dir = parts[-2]
        model = parts[-3]
        metric = parts[-4]

        rows.append({
            "metric": metric,
            "model": model,
            "group": "loca2",
            "scenario_dir": scenario_dir,
            "station": station,
            "file": f,
        })

    return pd.DataFrame(rows)


def load_completed_log():
    if not os.path.exists(LOG_JSON):
        return set()

    with open(LOG_JSON, "r") as f:
        return set(json.load(f))


def save_completed_log(completed):
    with open(LOG_JSON, "w") as f:
        json.dump(sorted(list(completed)), f, indent=2)


# ============================================================
# HISTORICAL GRIDMET
# ============================================================
def process_historical_gridmet(hist_nc):
    print("\nProcessing historical GRIDMET precipitation...")

    ds = open_nc_safe(hist_nc)

    try:
        pr_var = find_var(ds, GRIDMET_PR_CANDIDATES, "GRIDMET precipitation")
        print(f"Using GRIDMET precipitation variable: {pr_var}")

        pr = ds[pr_var]

        if "station" not in pr.dims:
            raise ValueError(
                f"{pr_var} must have a station dimension. Dims found: {pr.dims}"
            )

        records = []
        start_year, end_year = GRIDMET_PERIOD

        for stn in ds["station"].values:
            stn = str(stn)

            if stn in EXCLUDE_STATIONS:
                continue

            pr_stn = pr.sel(station=stn)
            hist_mask = get_period_mask(pr_stn["time"], start_year, end_year)
            hist_vals = monthly_total_climatology_values(pr_stn, hist_mask)

            if hist_vals is None:
                continue

            for month, val in zip(MONTHS, hist_vals):
                records.append({
                    "model": "gridmet",
                    "group": GRIDMET_GROUP,
                    "scenario_dir": "historical",
                    "station": stn,
                    "source": "historical",
                    "period": "historical",
                    "month": int(month),
                    "precip": float(val),
                })

        out = pd.DataFrame(records)

        if out.empty:
            raise ValueError("No historical GRIDMET records created.")

        print(f"Created {len(out)} historical GRIDMET rows.")
        return out

    finally:
        ds.close()


# ============================================================
# LOCA2 PROJECTIONS
# ============================================================
def process_loca2_projection_file(row):
    ds = open_nc_safe(row.file)

    try:
        pr_var = find_var(ds, LOCA2_PR_CANDIDATES, "LOCA2 precipitation")
        pr = ds[pr_var]

        records = []

        for period_name, (start_year, end_year) in PERIODS.items():
            mask = get_period_mask(pr["time"], start_year, end_year)
            vals = monthly_total_climatology_values(pr, mask)

            if vals is None:
                continue

            for month, val in zip(MONTHS, vals):
                records.append({
                    "model": row.model,
                    "group": row.group,
                    "scenario_dir": row.scenario_dir,
                    "station": row.station,
                    "source": "projection",
                    "period": period_name,
                    "month": int(month),
                    "precip": float(val),
                })

        return pd.DataFrame(records)

    finally:
        ds.close()


def process_loca2_projections(stations, overwrite_filelevel=False):
    file_table = list_loca2_projection_files(stations)

    print(f"\nFound {len(file_table)} LOCA2 projection files.")

    if file_table.empty:
        return pd.DataFrame()

    completed = set() if overwrite_filelevel else load_completed_log()
    all_records = []

    for row in tqdm(
        file_table.itertuples(index=False),
        total=len(file_table),
        desc="Processing LOCA2 projections"
    ):
        file_key = row.file

        if file_key in completed:
            continue

        try:
            df = process_loca2_projection_file(row)

            if not df.empty:
                all_records.append(df)

            completed.add(file_key)
            save_completed_log(completed)

        except Exception as e:
            print(f"Skipping {row.file}: {repr(e)}")

    if len(all_records) == 0:
        return pd.DataFrame()

    return pd.concat(all_records, ignore_index=True)


# ============================================================
# SUMMARY NETCDF
# ============================================================
def summarize_to_netcdf(filelevel_csv, out_nc):
    df = pd.read_csv(filelevel_csv)

    summary = (
        df.groupby(["source", "group", "period", "month"], as_index=False)
        .agg(
            mean=("precip", "mean"),
            median=("precip", "median"),
            q25=("precip", lambda x: np.nanpercentile(x, 25)),
            q75=("precip", lambda x: np.nanpercentile(x, 75)),
            min=("precip", "min"),
            max=("precip", "max"),
            n=("precip", "count"),
            n_models=("model", lambda x: x.nunique()),
            n_stations=("station", lambda x: x.nunique()),
        )
    )

    ds = (
        summary.set_index(["source", "group", "period", "month"])
        .to_xarray()
        .sortby("month")
    )

    ds.attrs["description"] = (
        "Monthly precipitation total climatology. "
        "Historical period uses GRIDMET. Projection periods use LOCA2."
    )
    ds.attrs["projection_root"] = PROJ_ROOT
    ds.attrs["historical_file"] = HIST_NC
    ds.attrs["historical_source"] = "GRIDMET"
    ds.attrs["projection_source"] = "LOCA2"
    ds.attrs["historical_period"] = f"{GRIDMET_PERIOD[0]}-{GRIDMET_PERIOD[1]}"
    ds.attrs["projection_periods"] = ", ".join(
        [f"{k}:{v[0]}-{v[1]}" for k, v in PERIODS.items()]
    )

    ds.to_netcdf(out_nc, engine="netcdf4")


# ============================================================
# MAIN
# ============================================================
def build_precip_monthly_totals(
    overwrite_summary=True,
    overwrite_filelevel=False
):
    if overwrite_filelevel and os.path.exists(FILELEVEL_CSV):
        print(f"Removing existing filelevel CSV: {FILELEVEL_CSV}")
        os.remove(FILELEVEL_CSV)

    if overwrite_filelevel and os.path.exists(LOG_JSON):
        print(f"Removing existing completed log: {LOG_JSON}")
        os.remove(LOG_JSON)

    stations = get_station_list(HIST_NC)

    print(f"Stations used ({len(stations)}):")
    print(stations)

    # --------------------------------------------------------
    # Historical GRIDMET
    # --------------------------------------------------------
    gridmet_df = process_historical_gridmet(HIST_NC)

    # --------------------------------------------------------
    # LOCA2 projections
    # --------------------------------------------------------
    loca2_df = process_loca2_projections(
        stations=stations,
        overwrite_filelevel=overwrite_filelevel
    )

    # --------------------------------------------------------
    # Merge with existing filelevel CSV
    # --------------------------------------------------------
    pieces = []

    if os.path.exists(FILELEVEL_CSV) and not overwrite_filelevel:
        old_df = pd.read_csv(FILELEVEL_CSV)

        old_df = old_df[
            ~(
                (old_df["source"] == "historical")
                & (old_df["group"] == GRIDMET_GROUP)
            )
        ].copy()

        pieces.append(old_df)

    pieces.append(gridmet_df)

    if not loca2_df.empty:
        pieces.append(loca2_df)

    new_df = pd.concat(pieces, ignore_index=True)

    new_df = new_df.drop_duplicates(
        subset=[
            "model",
            "group",
            "scenario_dir",
            "station",
            "source",
            "period",
            "month",
        ],
        keep="last"
    )

    new_df.to_csv(FILELEVEL_CSV, index=False)

    print(f"\nSaved filelevel CSV: {FILELEVEL_CSV}")

    # --------------------------------------------------------
    # Summary NetCDF
    # --------------------------------------------------------
    if overwrite_summary:
        print("\nBuilding summary NetCDF...")
        summarize_to_netcdf(FILELEVEL_CSV, SUMMARY_NC)
        print(f"Saved summary NC: {SUMMARY_NC}")

    print("\nDone.")
    print(f"Filelevel CSV: {FILELEVEL_CSV}")
    print(f"Summary NC:    {SUMMARY_NC}")


if __name__ == "__main__":
    build_precip_monthly_totals(
        overwrite_summary=True,
        overwrite_filelevel=False
    )


Processing historical GRIDMET precipitation...
Using GRIDMET precipitation variable: Gridmet_pr
Created 204 historical GRIDMET rows.
Updated existing filelevel CSV with GRIDMET historical: /data/keeling/a/tahsina2/Alam_et_al_2026/data/processed_projection_stats_precip_loca2_zarr/precip_monthly_totals_filelevel_rmse.csv

Building summary NetCDF from updated filelevel CSV...
Saved summary: /data/keeling/a/tahsina2/Alam_et_al_2026/data/processed_projection_stats_precip_loca2_zarr/precip_monthly_totals_with_historical_rmse_gridmet.nc

Done.
Filelevel CSV: /data/keeling/a/tahsina2/Alam_et_al_2026/data/processed_projection_stats_precip_loca2_zarr/precip_monthly_totals_filelevel_rmse.csv
Summary NC:    /data/keeling/a/tahsina2/Alam_et_al_2026/data/processed_projection_stats_precip_loca2_zarr/precip_monthly_totals_with_historical_rmse_gridmet.nc
